## Create Agent - Prebuit

### Build an agent with tools

In [ ]:
from dotenv import load_dotenv

_ = load_dotenv()

In [ ]:
# automatically reload all modules before executing new code. The captures changes in
# local packages.
%load_ext autoreload
%autoreload 2

In [ ]:
from typing import Annotated, List, Literal, Union

from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState
from langgraph.types import Command

@tool
def calculator(
    operation: Literal["add", "subtract", "multiply", "divide"],
    a: Union[int, float],
    b: Union[int, float],
) -> Union[int, float]:
    """Define a two-input calculator tool that returns precise answers
    
    Arg:
        operation (str): The operation to perform ('add', 'subtract', 'multiply', 'divide').
        a (float or int): The first number.
        b (float or int): The second number.

    Returns:
        result (float or int): the result of the operation
    Example
        Divide: result   = a / b
        Subtract: result = a - b
    """

    if operation == 'divide' and b == 0:
        return {"error": "Division by zero is not allowed."}

    # perform calculation
    if operation == "add":
        result = a + b
    elif operation == "subtract":
        result = a - b
    elif operation == "multiply":
        result = a * b
    elif operation == "divide":
        result == a / b
    else:
        result = "unknown operation" 

    return result

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from utils import format_messages

# create agent using create_agent directly
SYSTEM_PROMPT = """
You are a helpful arithmetic assistant who is an expert at using a calculator.
Return all text as plain text without Markdown math delimiters.
"""

model = init_chat_model(model="openai:gpt-4.1-mini", temperature=0.0)
tools = [calculator]

# create agente
agent = create_agent(
    model,
    tools,
    system_prompt=SYSTEM_PROMPT
).with_config({"recursion_limit": 20})

agent

In [ ]:
# create_agent returns a compiled graph
type(agent)

In [ ]:
# example usage
result1 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is 3.1 * 4.2?"
            }
        ]
    }
)

format_messages(result1["messages"])

In [ ]:
from IPython.display import JSON
from langchain_core.messages import messages_to_dict

JSON({"messages": messages_to_dict(result1["messages"])})

In [ ]:
from langchain.agents import AgentState

def reduce_list(left: list | None, right: list | None) -> list:
    """Safely combine two lists, handling cases where either or both inputs might be None
    
    Args:
        left (list | None): The first list to combine, or None.
        right (list | None): The second list to combine, or None.
        
    Returns:
        list: A new list containing all elements from both input lists.
              If a input is None, it's treated as an empty list.
    """

    if not left:
        left = []
    if not right:
        right = []
        
    return left + right

class CalcState(AgentState):
    """Graph State."""
    ops: Annotated[List[str], reduce_list]

In [ ]:
from typing import Annotated, List, Literal, Union

from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState
from langgraph.types import Command

@tool
def calculator_wstate(
    operation: Literal["add", "subtract", "multiply", "divide"],
    a: Union[int, float],
    b: Union[int, float],
    state: Annotated[CalcState, InjectedState], # not sent to LLM
    tool_call_id: Annotated[str, InjectedToolCallId] # not sent to LLM
) -> Union[int, float]:
    """Define a two-input calculator tool that returns precise answers
    
    Arg:
        operation (str): The operation to perform ('add', 'subtract', 'multiply', 'divide').
        a (float or int): The first number.
        b (float or int): The second number.

    Returns:
        result (float or int): the result of the operation
    Example
        Divide: result   = a / b
        Subtract: result = a - b
    """

    if operation == 'divide' and b == 0:
        return {"error": "Division by zero is not allowed."}

    # perform calculation
    if operation == "add":
        result = a + b
    elif operation == "subtract":
        result = a - b
    elif operation == "multiply":
        result = a * b
    elif operation == "divide":
        result == a / b
    else:
        result = "unknown operation" 

    ops = [f"({operation}, {a}, {b})"]

    return Command(
        update={
            "ops": ops,
            "messages": [
                ToolMessage(f"{result}", tool_call_id=tool_call_id)
            ]
        }
    )

In [ ]:
SYSTEM_PROMPT = """
You are a helpful arithmetic assistant who is an expert at using a calculator.
Return all text as plain text without Markdown math delimiters.
"""

model = init_chat_model(model="openai:gpt-4o-mini", temperature=0.0)
tools = [calculator_wstate] # new tool

# create agent
agent = create_agent(
    model,
    tools,
    system_prompt=SYSTEM_PROMPT,
    state_schema=CalcState # now defining state scheme
).with_config({"recursion_limit": 20}) # recursion_limit limits the number of steps the agent will run

In [ ]:
# example usage
result2 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is 3.1 * 4.2?"
            }
        ]
    }
)

format_messages(result2["messages"])

In [ ]:
JSON(result2)

In [ ]:
result3 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is 3.1 * 4.2 + 5.5 * 6.5?"
            }
        ]
    }
)

format_messages(result3["messages"])

In [ ]:
JSON(result3)

In [ ]:
JSON(result3["ops"])